In [ ]:
#| default_exp gnn_recommender

#| export
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv 
from neo4j import GraphDatabase
import numpy as np
import os

In [ ]:
#| export
from dotenv import load_dotenv
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

In [ ]:
#| export
def fetch_graph():

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        nodes = session.run("MATCH (n) RETURN id(n) as id, labels(n) as labels, n as props")
        node_list = [record for record in nodes]
        node_id_map = {rec['id']:idx for idx, rec in enumerate(node_list)}
        edges = session.run("MATCH (a)-[r]->(b) RETURN id(a) as src, id(b) as dst")
        edge_index = [[], []]
        for record in edges:
            edge_index[0].append(node_id_map[record['src']])
            edge_index[1].append(node_id_map[record['dst']])
        x = torch.eye(len(node_list))
        edge_index = torch.tensor(edge_index, dtype=torch.long)
        data = Data(x=x, edge_index=edge_index)
    driver.close()
    return data

In [ ]:
#| export
class SimpleGCN(torch.nn.Module):

    def __init__(self, num_features, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 2)
    
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x ,edge_index)
        return x

In [ ]:
#| export
data = fetch_graph()
model = SimpleGCN(num_features=data.num_features, hidden_channels=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
model.train()

for epoch in range(50):
    optimizer.zero_grad()
    out = model(data)

    y = torch.zeros(data.num_nodes, dtype=torch.long)
    loss = F.cross_entropy(out, y)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")
    print("Training complete.")



Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function. ('id' has been replaced by 'elementId or an application-generated id')} {position: line: 1, column: 18, offset: 17} for query: 'MATCH (n) RETURN id(n) as id, labels(n) as labels, n as props'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function. ('id' has been replaced by 'elementId or an application-generated id')} {position: line: 1, column: 27, offset: 26} for query: 'MATCH (a)-[r]->(b) RETURN id(a) as src, id(b) as dst'
Received notification from DBMS server: {severity: WARNING} {code: Ne

Epoch 0, Loss: 0.3879380524158478
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Epoch 10, Loss: 0.10166361182928085
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Epoch 20, Loss: 0.027216127142310143
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Epoch 30, Loss: 0.009356711059808731
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Training complete.
Epoch 40, Loss: 0.0039712050929665565
Training complete.
Training complete.
Training complete.
T

In [ ]:
#| export
torch.save(model.state_dict(), "../model/recommender_model.pth")